# 05 · Joining tables (JOIN)

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~45 min**

## Learning objectives
- combine tables with `INNER JOIN ... ON`;
- join three tables (`abundance` ↔ `samples` ↔ `taxa`);
- keep unmatched rows with `LEFT JOIN` (and spot NULLs).

Real questions need columns from several tables at once. A **JOIN** combines rows from two tables that share a key. This is the heart of relational SQL.

---
| # | Lesson |
|---|---|
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Hands-on capstone |
| 08 | Bonus: creating & changing data |

## ⚙️ Setup — run this cell first

This connects the notebook to the example database. It works both on **your own computer** and on **Google Colab** — just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

## 1. Join abundance to taxa
The `abundance` table only stores IDs. Join `taxa` to get the phylum, then total the reads per phylum.

In [ ]:
%%sql
SELECT t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.phylum
ORDER BY reads DESC;

## 2. Join abundance to samples
Which environment has the most reads?

In [ ]:
%%sql
SELECT s.environment, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
GROUP BY s.environment
ORDER BY reads DESC;

## 3. Three-table join — the payoff
Phylum composition **per environment**: this recovers the ecology (Cyanobacteria in freshwater, methanogens in sediment…).

In [ ]:
%%sql
SELECT s.environment, t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
GROUP BY s.environment, t.phylum
ORDER BY s.environment, reads DESC;

## 4. Bring in the sites table
Reads per country (four-table reach via keys).

In [ ]:
%%sql
SELECT si.country, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN sites si  ON s.site = si.site
GROUP BY si.country
ORDER BY reads DESC;

## 5. LEFT JOIN — keep rows with no match
List every taxon and its count in sample `S001`; taxa absent from S001 show `NULL` (an INNER JOIN would drop them).

In [ ]:
%%sql
SELECT t.genus, a.count
FROM taxa t
LEFT JOIN abundance a ON t.taxon_id = a.taxon_id AND a.sample_id = 'S001'
ORDER BY a.count DESC;

---
## Exercises

**Exercise.** Total reads per genus (join abundance to taxa); show the top 10.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT t.genus, SUM(a.count) AS reads
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.genus
ORDER BY reads DESC
LIMIT 10;
```
</details>

**Exercise.** Reads per environment for the domain `Archaea` only (join all three, filter on domain).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT s.environment, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
WHERE t.domain = 'Archaea'
GROUP BY s.environment
ORDER BY reads DESC;
```
</details>

**Exercise.** Which single phylum has the most reads in `Sediment` samples?

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
WHERE s.environment = 'Sediment'
GROUP BY t.phylum
ORDER BY reads DESC
LIMIT 1;
```
</details>

### Recap
- `JOIN b ON a.key = b.key` combines tables that share a key.
- Chain multiple `JOIN`s to reach three or four tables.
- `LEFT JOIN` keeps unmatched left-side rows (their right columns are NULL).
- Give tables short aliases (`a`, `s`, `t`) and prefix columns.
- Next: **06 · Subqueries & CTEs**.